In [ ]:
import os
import numpy as np
import pandas as pd
from wfdb import rdsamp
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

dataset_path = Path("/opt/gpudata/ecg/zzu-pecg")

# NOTE: it's hard to construct intrinsic (e.g. time-defined) train/val/test splits
# from the source data while preserving label presence in all splits if we
# use fine-grained labels defined below, hence reliance on coarser labels - 
# BUT we do ensure the presence of each fine-grain label across the train subsets
fine_grained_labels = {
    "Fulminant myocarditis": ["(F) I40.0"],
    "Viral myocarditis": ["(V) I40.0"],
    "Acute myocarditis": ["I40.9"],
    "_Myocarditis": ["I51.4"],
    "Dilated cardiomypoathy": ["I42.0"],
    "Hypertrophic cardiomyopathy": ["I42.2"],
    "_Cardiomyopathy": ["I42.9"],
    "Noncompaction of the ventricular myocardium": ["Q24.8"],
    "_Kawasaki disease": ["M30.3"],
    "Ventricular septal defect": ["Q21.0"],
    "Atrial septal defect": ["Q21.1"],
    "Atrial septal defect (Foramen ovale)": ["(FO) Q21.1"],
    "Atrial septal defect (Ostium secundum defect)": ["(OSD) Q21.1"],
    "Atrioventricular septal defect": ["Q21.2"],
    "Tetralogy of Fallot": ["Q21.3"],
    "Stenosis of right ventricular outflow tract": ["Q22.1"],
    "Patent ductus arteriosus": ["Q25.0"],
    "Pulmonary stenosis": ["Q25.6"],
    "Pulmonary valve stenosis": ["I37.0"],
}
coarse_grained_labels = {
    "Myocarditis": ["(F) I40.0", "(V) I40.0", "I40.9", "I51.4"],
    "Cardiomyopathy": ["I42.0", "I42.2", "I42.9", "Q24.8"],
    "Kawasaki disease": ["M30.3"],
    "Congenital heart disease": [
        "Q21.0",
        "Q21.1",
        "(FO) Q21.1",
        "(OSD) Q21.1",
        "Q21.2",
        "Q21.3",
        "Q22.1",
        "Q25.0",
        "Q25.6",
        "I37.0",
    ],
}

In [ ]:
df = pd.read_csv(dataset_path / "AttributesDictionary.csv")
df = df[df["Lead"] == 12].sort_values(["Patient_ID", "ECG_ID"]).reset_index(drop=True)
original_cols = df.columns

In [ ]:
# one-hot encode labels
dxs = df["ICD-10 code"].str.split(";").explode() # create a long table of idx --> icd code, where idx can appear multiple time
dx_idxs = {icd.strip("'"): list(group.index) for icd, group in dxs.groupby(dxs)} # convert to dict of icd code --> list[idx]

# convert to wide
for label_map in [fine_grained_labels, coarse_grained_labels]:
    for label, icds in label_map.items():
        df[label] = 0
        for icd in icds:
            df.loc[dx_idxs[icd], label] = 1

In [ ]:
# create intrinsic data splits based on time
train_val_split_date = "2023"
val_test_split_date = "2023-08"

# require first ECG per patient in val/test
assert (df.sort_values(["Patient_ID", "ECG_ID"]).index == df.index).all()
first_ecg_mask = ~df.duplicated("Patient_ID", keep="first")

train_mask = df["Acquisition_date"] < train_val_split_date
val_mask = df["Acquisition_date"].between(train_val_split_date, val_test_split_date) & first_ecg_mask
test_mask = (df["Acquisition_date"] > val_test_split_date) & first_ecg_mask

# check that all coarse grained labels present in val/test
assert (df.loc[val_mask, list(coarse_grained_labels)].sum() > 0).all()
assert (df.loc[test_mask, list(coarse_grained_labels)].sum() > 0).all()

df["split"] = "no_split"
df.loc[train_mask, "split"] = "train"
df.loc[val_mask, "split"] = "val"
df.loc[test_mask, "split"] = "test"

In [ ]:
df["split"].value_counts()

In [ ]:
# make sure each fine-grained target is represented at least once in each train subset
train_df = df[df["split"].isin(["train"])].reset_index(drop=True)
val_test_df = df[df["split"].isin(["val", "test"])].reset_index(drop=True)
rng = np.random.default_rng(42)
selected_ecgs = set()
for label_col in fine_grained_labels:
    ecg_id = rng.choice(train_df.loc[train_df[label_col] == 1, "ECG_ID"], 1)
    selected_ecgs.add(ecg_id.item())
print(f"selected {len(selected_ecgs)} ECGs for {len(fine_grained_labels)} fine-grained labels")

In [ ]:
# exclude selected ECGs from subset, add in after
mask = train_df["ECG_ID"].isin(selected_ecgs)
curr_df = train_df[~mask]
sel_df = train_df[mask]

# make nested, stratified subsets according to age/sex
assert train_df["Age"].str.endswith("d").all()
age_bin = pd.qcut(train_df["Age"].str[:-1].astype(int), q=4).cat.codes
male_sex = (train_df["Gender"].str.strip("'") == "Male").astype(int)
splitter = age_bin.astype(str) + "_" + male_sex.astype(str)
assert (splitter.value_counts() >= 2).all()

train_subsets = [curr_df]
for tgt_size in [4096, 2048, 1024, 512, 256]:
    select_size = tgt_size - len(selected_ecgs)
    curr_df = train_subsets[-1]
    _, next_df = train_test_split(curr_df, test_size=select_size, stratify=splitter.loc[curr_df.index], random_state=42)
    train_subsets.append(next_df)

# add back in selected ECGs to ensure each label has at least 1 training example
train_subsets = [pd.concat([x, sel_df], ignore_index=True) for x in train_subsets]

# check that all labels are present in all subsets
assert all([(subset.loc[:, list(fine_grained_labels)].sum() > 0).all() for subset in train_subsets])

In [ ]:
# write subsets to disk
files_to_link = ["Child_ecg"]
subsets = [pd.concat([x, val_test_df], ignore_index=True).sort_values(["Patient_ID", "ECG_ID"]) for x in train_subsets]
for subset, suffix in zip(subsets, ["", "-4k", "-2k", "-1k", "-512", "-256"]):
    subset_path = dataset_path.parent / f"{dataset_path.name}{suffix}"
    os.makedirs(subset_path, exist_ok=True)
    subset[original_cols].to_csv(os.path.join(subset_path, "AttributesDictionary.csv"), index=False)
    for fname in files_to_link:
        link_name = os.path.join(subset_path, fname)
        if not os.path.exists(link_name):
            os.symlink(dataset_path / fname, link_name)

In [ ]:
# compute waveform stats:
# because ZZU ECGs are variable length, can't stack directly
# however, stats are computed along time dim as well, so we can
# just concenate all ECGs end to end before computing stats
train_df["fpath"] = dataset_path / "Child_ecg" / train_df["Filename"]
X = []
for i, fpath in enumerate(tqdm(train_df["fpath"])):
    x, _ = rdsamp(fpath)
    n_timesteps, n_leads = x.shape
    assert n_leads == 12
    assert train_df.loc[i, "Sampling_point"] == n_timesteps
    assert not np.isnan(x).any()
    X.append(x)
X = np.concatenate(X)

In [ ]:
lowers, uppers = np.percentile(X, [0.1, 99.9], axis=0)
display(lowers.tolist())
display(uppers.tolist())

In [ ]:
X_clipped = np.clip(X, lowers, uppers)
means = X_clipped.mean(axis=0)
stds = X_clipped.std(axis=0)
display(means.tolist())
display(stds.tolist())